# Lab 1: OpenAI API Fundamentals — SOLUTIONS
## CCI Session 3 — Data Science Foundations

**Duration:** 15 minutes
**Mode:** Guided step-by-step

### Clinical Scenario
> You are building the first component of a clinical AI pipeline at KHCC. Your task is to connect to the OpenAI API from Python, understand how tokens and costs work, and apply different prompting techniques to classify clinical text.

### Objective
By the end of this lab, you will:
- Make API calls to GPT models from Python
- Count tokens and estimate costs
- Apply zero-shot, few-shot, and chain-of-thought prompting

## Setup — Install and Import

In [ ]:
# Run this cell first to install required packages
!pip install -q openai tiktoken

import openai
from openai import OpenAI
import tiktoken
import json
import os

## Cell 1 — API Key Setup
Enter your OpenAI API key. Never hardcode keys in shared notebooks!

In [ ]:
# === CELL 1: API KEY ===
from google.colab import userdata
api_key = userdata.get('OPENAI_API_KEY')

# Create the OpenAI client
client = OpenAI(api_key=api_key)
print("Client created successfully!")

## Cell 2 — Your First API Call
Make a simple chat completion call. Ask the model to classify a clinical note as one of:
**Discharge Summary, Progress Note, Consultation Note, Pathology Report.**

In [ ]:
# === CELL 2: FIRST API CALL ===
clinical_note = """
Patient was seen in the oncology clinic for follow-up after completing 
4 cycles of AC chemotherapy for Stage IIA breast cancer. No new complaints.
Physical exam unremarkable. Plan to start paclitaxel weekly x12.
"""

response = client.chat.completions.create(
    model="gpt-4o-mini",
    temperature=0.0,
    messages=[
        {
            "role": "system",
            "content": "You are a clinical document classifier. Classify the note as one of: Discharge Summary, Progress Note, Consultation Note, Pathology Report. Return only the classification."
        },
        {
            "role": "user",
            "content": clinical_note
        }
    ]
)

# Extract and print the response text
classification = response.choices[0].message.content
print(f"Classification: {classification}")
print(f"\nModel used: {response.model}")
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Total tokens: {response.usage.total_tokens}")

## Cell 3 — Token Counting with tiktoken
Count tokens BEFORE sending to estimate costs and check context limits.

In [ ]:
# === CELL 3: TOKEN COUNTING ===

# Create a tiktoken encoder for the model
encoding = tiktoken.encoding_for_model("gpt-4o-mini")

# Count tokens in the clinical_note
tokens = encoding.encode(clinical_note)
print(f"Number of tokens: {len(tokens)}")
print(f"First 10 token IDs: {tokens[:10]}")
print(f"Decoded first 10 tokens: {[encoding.decode([t]) for t in tokens[:10]]}")

# Calculate estimated cost
# GPT-4o-mini pricing: $0.15 per 1M input tokens, $0.60 per 1M output tokens
input_tokens = len(tokens)
output_tokens = 20  # approximate response length

input_cost = (input_tokens / 1_000_000) * 0.15
output_cost = (output_tokens / 1_000_000) * 0.60
total_cost = input_cost + output_cost

print(f"\nEstimated cost:")
print(f"  Input:  ${input_cost:.8f}")
print(f"  Output: ${output_cost:.8f}")
print(f"  Total:  ${total_cost:.8f}")
print(f"\nAt this rate, you could process ~{int(1.0 / total_cost):,} notes for $1.00")

## Cell 4 — Temperature Experiment
Run the same prompt 3 times with `temperature=0.0`, then 3 times with `temperature=1.5`.
Compare consistency.

In [ ]:
# === CELL 4: TEMPERATURE EXPERIMENT ===
prompt = "List 3 possible side effects of cisplatin chemotherapy."

# Run the prompt 3 times with temperature=0.0
deterministic_results = []
for i in range(3):
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0.0,
        messages=[{"role": "user", "content": prompt}]
    )
    deterministic_results.append(resp.choices[0].message.content)

# Run the prompt 3 times with temperature=1.5
random_results = []
for i in range(3):
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=1.5,
        messages=[{"role": "user", "content": prompt}]
    )
    random_results.append(resp.choices[0].message.content)

# Print and compare
print("=" * 60)
print("DETERMINISTIC (temperature=0.0)")
print("=" * 60)
for i, result in enumerate(deterministic_results):
    print(f"\n--- Run {i+1} ---")
    print(result)

print("\n" + "=" * 60)
print("RANDOM (temperature=1.5)")
print("=" * 60)
for i, result in enumerate(random_results):
    print(f"\n--- Run {i+1} ---")
    print(result)

# Check if deterministic results are identical
print("\n" + "=" * 60)
print(f"Deterministic results identical: {deterministic_results[0] == deterministic_results[1] == deterministic_results[2]}")
print(f"Random results identical: {random_results[0] == random_results[1] == random_results[2]}")

## Cell 5 — Prompting Techniques
Apply **zero-shot**, **few-shot**, and **chain-of-thought** to the same clinical task:
extracting the cancer type from a clinical note.

In [ ]:
# === CELL 5: PROMPTING TECHNIQUES ===
test_note = "Patient is a 62-year-old male presenting with a 3cm mass in the sigmoid colon, biopsy-confirmed adenocarcinoma. CEA elevated at 8.2. CT shows no distant metastases. Plan: surgical resection followed by adjuvant FOLFOX."

# --- Zero-Shot ---
print("=== ZERO-SHOT ===")
zero_shot = client.chat.completions.create(
    model="gpt-4o-mini",
    temperature=0.0,
    messages=[
        {"role": "system", "content": "You are a clinical data extractor."},
        {"role": "user", "content": f"What is the cancer type in this note?\n\n{test_note}"}
    ]
)
print(zero_shot.choices[0].message.content)

# --- Few-Shot ---
print("\n=== FEW-SHOT ===")
few_shot = client.chat.completions.create(
    model="gpt-4o-mini",
    temperature=0.0,
    messages=[
        {"role": "system", "content": "You are a clinical data extractor. Extract the cancer type from clinical notes. Use the format: Organ cancer - Histological type"},
        {"role": "user", "content": "52F with ER+ invasive ductal carcinoma of the left breast"},
        {"role": "assistant", "content": "Breast cancer - Invasive ductal carcinoma"},
        {"role": "user", "content": "71M diagnosed with small cell lung cancer, extensive stage"},
        {"role": "assistant", "content": "Lung cancer - Small cell"},
        {"role": "user", "content": test_note}
    ]
)
print(few_shot.choices[0].message.content)

# --- Chain-of-Thought ---
print("\n=== CHAIN-OF-THOUGHT ===")
cot = client.chat.completions.create(
    model="gpt-4o-mini",
    temperature=0.0,
    messages=[
        {"role": "system", "content": "You are a clinical data extractor."},
        {"role": "user", "content": f"Extract the cancer type from this note. Think step by step:\n1) Identify the organ\n2) Identify the histology\n3) State the cancer type in format: Organ cancer - Histological type\n\n{test_note}"}
    ]
)
print(cot.choices[0].message.content)

## Cell 6 — Streaming Responses
For long outputs, stream tokens as they arrive.

In [ ]:
# === CELL 6: STREAMING ===
print("Streaming response:\n")

stream = client.chat.completions.create(
    model="gpt-4o-mini",
    temperature=0.7,
    stream=True,
    messages=[
        {"role": "system", "content": "You are a patient education specialist."},
        {"role": "user", "content": "Write a brief patient education summary about managing nausea during chemotherapy."}
    ]
)

full_response = ""
for chunk in stream:
    if chunk.choices[0].delta.content:
        content = chunk.choices[0].delta.content
        print(content, end="", flush=True)
        full_response += content

print(f"\n\n--- Stream complete. Total characters: {len(full_response)} ---")

## Stretch Challenge
Build a function `classify_note(note_text)` that takes any clinical note, counts tokens,
makes an API call, and returns both the classification and the cost estimate.

In [ ]:
# === STRETCH CHALLENGE ===
def classify_note(note_text):
    """
    Classify a clinical note and return the classification with cost estimate.
    
    Args:
        note_text (str): The clinical note text
    
    Returns:
        dict: {"classification": str, "input_tokens": int, "output_tokens": int, "estimated_cost_usd": float}
    """
    # Count tokens
    encoding = tiktoken.encoding_for_model("gpt-4o-mini")
    input_token_count = len(encoding.encode(note_text))
    
    # Make API call
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0.0,
        messages=[
            {
                "role": "system",
                "content": "You are a clinical document classifier. Classify the note as one of: Discharge Summary, Progress Note, Consultation Note, Pathology Report. Return only the classification."
            },
            {"role": "user", "content": note_text}
        ]
    )
    
    # Extract results
    classification = response.choices[0].message.content
    output_tokens = response.usage.completion_tokens
    actual_input_tokens = response.usage.prompt_tokens
    
    # Calculate cost
    cost = (actual_input_tokens / 1_000_000) * 0.15 + (output_tokens / 1_000_000) * 0.60
    
    return {
        "classification": classification,
        "input_tokens": actual_input_tokens,
        "output_tokens": output_tokens,
        "estimated_cost_usd": cost
    }

# Test it
result = classify_note(clinical_note)
print(json.dumps(result, indent=2))

## Expected Outputs

| Cell | What You Should See |
|------|--------------------|
| Cell 2 | `Progress Note` |
| Cell 3 | Token count ~45-55, cost < $0.001 |
| Cell 4 | temperature=0.0 gives identical outputs; temperature=1.5 gives varied outputs |
| Cell 5 | All three techniques identify "Colorectal cancer - Adenocarcinoma" (CoT gives reasoning) |
| Cell 6 | Tokens stream in real-time |

---

### KHCC Connection
This lab mirrors the first step in the **AIDI clinical AI pipeline**: connecting to LLM APIs
from Python, understanding costs at scale, and choosing the right prompting strategy for
clinical NLP tasks like note classification and entity extraction.